# AI Programming — Lecture 3
## 선형 회귀 (Linear Regression)

이 노트북에서는 **선형 회귀를 직접 계산하는 방법부터 TensorFlow/Keras 구현까지** 단계적으로 실습합니다.

### 학습 목표
실습을 마치면 다음 내용을 설명하고 구현할 수 있어야 합니다.

- 선형 회귀 모델 $\hat{y}=ax+b$의 의미를 설명할 수 있습니다.
- 최소제곱법(Least Squares Method)으로 최적의 직선을 구할 수 있습니다.
- 원소별 계산과 벡터화(vectorized) 계산의 차이를 이해합니다.
- MAE와 MSE를 계산하고 차이를 설명할 수 있습니다.
- 경사하강법(Gradient Descent)으로 모델 파라미터를 학습할 수 있습니다.
- 학습률(learning rate)이 학습에 미치는 영향을 관찰할 수 있습니다.
- 여러 입력 변수를 사용하는 다중 선형 회귀를 구현할 수 있습니다.
- TensorFlow/Keras로 동일한 문제를 구현할 수 있습니다.

### 실습 방법
1. 셀을 **위에서부터 순서대로 실행**하세요.
2. 각 절의 **확인할 내용**을 읽고 결과를 관찰하세요.
3. `직접 해보기`가 표시된 부분에서는 값을 변경한 뒤 다시 실행해 보세요.
4. 코드가 예상과 다르게 동작하면 바로 다음 셀로 넘어가기보다 출력값과 그래프를 먼저 확인하세요.

> 그래프의 축과 제목은 실행 환경의 한글 폰트 문제를 피하기 위해 일부 영어로 표시합니다.


## 0. 라이브러리 불러오기

이번 실습에서는 `NumPy`를 이용해 수치 계산을 수행하고, `Matplotlib`으로 결과를 시각화합니다.

아래 셀을 먼저 실행하세요.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 출력되는 숫자의 소수점 자릿수를 보기 좋게 설정
np.set_printoptions(precision=4, suppress=True)


## 1. 데이터 살펴보기

먼저 매우 작은 예제를 이용해 선형 회귀의 동작을 확인합니다.

| 공부 시간 (시간) | 1 | 3 | 5 | 7 | 9 |
|---|---:|---:|---:|---:|---:|
| 시험 점수 | 75 | 77 | 85 | 83 | 90 |

선형 회귀 모델은 다음과 같이 표현합니다.

$$
\hat{y} = ax + b
$$

- $x$: 입력값
- $\hat{y}$: 모델의 예측값
- $a$: 기울기(slope)
- $b$: 절편(intercept)

### 확인할 내용
아래 산점도를 보고 **공부 시간이 증가할수록 시험 점수가 대체로 증가하는지** 확인하세요.


In [ ]:
# 입력 데이터: 공부 시간
x = np.array([1, 3, 5, 7, 9], dtype=float)

# 정답 데이터: 시험 점수
y = np.array([75, 77, 85, 83, 90], dtype=float)

# 데이터 분포 확인
plt.scatter(x, y)
plt.xlabel("Study Time (hours)")
plt.ylabel("Exam Score")
plt.title("Study Time vs. Exam Score")
plt.grid(alpha=0.3)
plt.show()


### 1.1 임의의 직선으로 예측하기

먼저 $a$와 $b$를 직접 정해 예측값을 만들어 봅니다.

NumPy 배열을 사용하면 모든 데이터에 대한 예측을 한 줄로 계산할 수 있습니다.

$$
\hat{\mathbf{y}} = a\mathbf{x}+b
$$

### 확인할 내용
- `y_pred`와 실제값 `y`의 차이를 확인하세요.
- 오차가 모두 같은 방향인지, 양수와 음수가 섞여 있는지 살펴보세요.


In [ ]:
# 임의로 정한 직선의 파라미터
a = 2.0
b = 72.0

# 모든 데이터에 대해 한 번에 예측
y_pred = a * x + b

# 실제값 - 예측값
error = y - y_pred

print("예측값:", y_pred)
print("오차:", error)


## 2. 최소제곱법으로 최적의 직선 구하기

최소제곱법(Least Squares Method)은 **실제값과 회귀선 사이의 제곱 오차 합이 최소**가 되도록 파라미터를 구하는 방법입니다.

### 2.1 원소별 계산

단순 선형 회귀의 기울기와 절편은 다음과 같이 계산할 수 있습니다.

$$
a =
\frac{
\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})
}{
\sum_{i=1}^{n}(x_i-\bar{x})^2
}
$$

$$
b = \bar{y} - a\bar{x}
$$

아래 코드는 이 식을 그대로 구현합니다.

### 확인할 내용
`a_lsm`과 `b_lsm`이 어떤 값으로 계산되는지 확인하세요.


In [ ]:
# x와 y의 평균
mx = np.mean(x)
my = np.mean(y)

# 분모: Σ(x_i - x_mean)^2
down = sum([(i - mx) ** 2 for i in x])

# 분자: Σ(x_i - x_mean)(y_i - y_mean)
top = 0.0
for i in range(len(x)):
    top += (x[i] - mx) * (y[i] - my)

# 최소제곱법으로 기울기와 절편 계산
a_lsm = top / down
b_lsm = my - mx * a_lsm

print(f"회귀식: y = {a_lsm:.4f}x + {b_lsm:.4f}")


### 2.2 벡터화된 계산

같은 문제를 행렬 연산으로 표현할 수도 있습니다.

설계 행렬(design matrix)을 이용하면 정규방정식(normal equation)은 다음과 같습니다.

$$
\boldsymbol{\beta}
=
(\mathbf{X}^{\top}\mathbf{X})^{-1}
\mathbf{X}^{\top}\mathbf{y}
$$

여기서

$$
\boldsymbol{\beta}
=
\begin{bmatrix}
a \\
b
\end{bmatrix}
$$

입니다.

### 확인할 내용
- 원소별 계산으로 구한 `a_lsm`, `b_lsm`
- 행렬 연산으로 구한 `a_vec`, `b_vec`

두 결과가 거의 같은지 비교하세요.


In [ ]:
# 첫 번째 열: x
# 두 번째 열: 절편 b를 위한 1
X = np.column_stack((x, np.ones(len(x))))

# 정규방정식 β = (X^T X)^(-1) X^T y
beta = np.linalg.inv(X.T @ X) @ X.T @ y

a_vec, b_vec = beta

print("설계 행렬 X:")
print(X)
print(f"회귀식: y = {a_vec:.4f}x + {b_vec:.4f}")


### 2.3 최적 회귀선 시각화

최소제곱법으로 구한 파라미터를 이용해 회귀선을 그립니다.

### 확인할 내용
- 회귀선이 모든 데이터 점을 정확히 지나지는 않습니다.
- 대신 전체 데이터에 대한 **제곱 오차를 가장 작게 만드는 직선**입니다.


In [ ]:
y_lsm = a_vec * x + b_vec

plt.scatter(x, y, label="Data")
plt.plot(x, y_lsm, label="Least-squares line")
plt.xlabel("Study Time (hours)")
plt.ylabel("Exam Score")
plt.title("Linear Regression with LSM")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. 예측 오차 측정하기

각 데이터의 오차(error)는 실제값과 예측값의 차이입니다.

$$
e_i = y_i - \hat{y}_i
$$

회귀 문제에서는 오차를 하나의 숫자로 요약하기 위해 MAE와 MSE를 자주 사용합니다.

$$
\mathrm{MAE}
=
\frac{1}{n}\sum_{i=1}^{n}|e_i|
$$

$$
\mathrm{MSE}
=
\frac{1}{n}\sum_{i=1}^{n}e_i^2
$$

### MAE와 MSE의 차이
- **MAE**: 오차의 절댓값을 평균냅니다.
- **MSE**: 큰 오차를 제곱하기 때문에 큰 실수에 더 큰 벌점을 줍니다.

### 확인할 내용
같은 예측 결과에 대해 MAE와 MSE의 크기가 어떻게 다른지 확인하세요.


In [ ]:
# 최소제곱 회귀선의 예측값과 오차
error = y - y_lsm

# 대표적인 회귀 성능 지표
mae = np.mean(np.abs(error))
mse = np.mean(error ** 2)

print("예측값:", y_lsm)
print("오차:", error)
print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")


> ### ✅ 체크포인트
> 최소제곱법의 원소별 구현과 행렬 구현이 **같은 문제를 다른 방식으로 계산한 것**임을 확인했나요?

### 직접 해보기 1 — 회귀선을 바꾸어 보기

아래 코드에서 `a_try`와 `b_try` 값을 변경해 보세요.

예를 들어 다음을 시도해 볼 수 있습니다.

- `a_try = 0.0`, `b_try = 82.0`
- `a_try = 3.0`, `b_try = 65.0`
- 직접 선택한 값

### 질문
1. 어떤 경우에 MAE가 작아집니까?
2. 어떤 경우에 MSE가 크게 증가합니까?
3. 최소제곱법으로 구한 회귀선보다 더 작은 MSE를 만들 수 있습니까?


In [ ]:
# TODO: 아래 두 값을 바꾸어 보세요.
a_try = 1.0
b_try = 75.0

y_try = a_try * x + b_try

print("MAE:", np.mean(np.abs(y - y_try)))
print("MSE:", np.mean((y - y_try) ** 2))


## 4. 손실 함수의 모양 살펴보기

선형 회귀에서 MSE 손실은 $a$, $b$에 따라 달라집니다.

$$
\mathcal{L}(a,b)
=
\frac{1}{n}
\sum_{i=1}^{n}
\left(y_i-(ax_i+b)\right)^2
$$

아래에서는 다양한 $a$, $b$ 값을 대입하여 손실 함수의 등고선(contour)을 그립니다.

### 확인할 내용
그래프의 `Minimum` 표시가 앞에서 최소제곱법으로 구한 $(a,b)$와 일치하는지 확인하세요.

> 핵심 아이디어: **학습(training)은 손실이 작은 파라미터를 찾는 과정**으로 볼 수 있습니다.


In [ ]:
a_values = np.linspace(-2, 6, 120)
b_values = np.linspace(55, 90, 120)

A, B = np.meshgrid(a_values, b_values)
L = np.zeros_like(A)

for r in range(A.shape[0]):
    for c in range(A.shape[1]):
        pred = A[r, c] * x + B[r, c]
        L[r, c] = np.mean((y - pred) ** 2)

plt.figure(figsize=(7, 5))
contour = plt.contour(A, B, L, levels=25)
plt.clabel(contour, inline=True, fontsize=7)
plt.scatter([a_vec], [b_vec], marker="x", s=80, label="Minimum")
plt.xlabel("a")
plt.ylabel("b")
plt.title("MSE Loss Landscape")
plt.legend()
plt.show()

## 5. 경사하강법으로 파라미터 학습하기

최소제곱법은 해를 직접 계산하지만, 신경망에서는 보통 **경사하강법(Gradient Descent)**을 사용하여 파라미터를 반복적으로 갱신합니다.

MSE에 대한 gradient는 다음과 같습니다.

$$
\frac{\partial \mathcal{L}}{\partial a}
=
-\frac{2}{n}
\sum_{i=1}^{n}
x_i\left(y_i-(ax_i+b)\right)
$$

$$
\frac{\partial \mathcal{L}}{\partial b}
=
-\frac{2}{n}
\sum_{i=1}^{n}
\left(y_i-(ax_i+b)\right)
$$

파라미터는 gradient의 반대 방향으로 갱신합니다.

$$
a \leftarrow a - \eta \frac{\partial \mathcal{L}}{\partial a}
$$

$$
b \leftarrow b - \eta \frac{\partial \mathcal{L}}{\partial b}
$$

여기서 $\eta$는 **학습률(learning rate)**입니다.

### 확인할 내용
- epoch가 증가하면서 MSE가 감소하는지 확인하세요.
- 마지막에 얻은 $a$, $b$가 최소제곱법 결과와 가까운지 비교하세요.


In [ ]:
# 초기 파라미터
a = 0.0
b = 0.0

# 학습률과 반복 횟수
lr = 0.003
epochs = 8001
n = len(x)

loss_history = []

for epoch in range(epochs):
    # 1) 예측
    y_pred = a * x + b

    # 2) 오차
    error = y - y_pred

    # 3) gradient 계산
    a_diff = -(2 / n) * np.sum(x * error)
    b_diff = -(2 / n) * np.sum(error)

    # 4) 파라미터 업데이트
    a -= lr * a_diff
    b -= lr * b_diff

    # 5) 현재 손실 기록
    loss = np.mean(error ** 2)
    loss_history.append(loss)

    if epoch % 1000 == 0:
        print(f"epoch: {epoch:4d}, a: {a:.6f}, b: {b:.6f}, MSE: {loss:.6f}")

print(f"최종 모델: y = {a:.4f}x + {b:.4f}")


In [ ]:
y_pred = a * x + b

plt.scatter(x, y, label="Data")
plt.plot(x, y_pred, label="Gradient-descent line")
plt.xlabel("Study Time (hours)")
plt.ylabel("Exam Score")
plt.title("Linear Regression with Gradient Descent")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Training Loss")
plt.yscale("log")
plt.grid(alpha=0.3)
plt.show()

> ### ✅ 체크포인트
> 경사하강법으로 구한 회귀선과 최소제곱법으로 구한 회귀선을 비교해 보세요.  
> 두 방법의 **계산 방식은 다르지만 같은 최적점에 접근**한다는 점이 핵심입니다.

### 직접 해보기 2 — 학습률 비교

학습률은 한 번의 업데이트에서 파라미터를 얼마나 크게 이동할지 결정합니다.

아래 코드에서는 세 가지 학습률을 비교합니다.

### 예상해 보기
코드를 실행하기 전에 다음을 생각해 보세요.

- 너무 작은 학습률은 어떤 문제가 있을까요?
- 너무 큰 학습률은 어떤 문제가 있을까요?

### 확인할 내용
각 학습률에 대해 최종 MSE와 학습된 파라미터를 비교하세요.


In [ ]:
def train_linear_regression(lr, epochs=1000):
    """주어진 learning rate로 단순 선형 회귀를 학습합니다."""

    a = 0.0
    b = 0.0
    n = len(x)
    history = []

    for _ in range(epochs):
        y_pred = a * x + b
        error = y - y_pred

        a_diff = -(2 / n) * np.sum(x * error)
        b_diff = -(2 / n) * np.sum(error)

        a -= lr * a_diff
        b -= lr * b_diff

        loss = np.mean(error ** 2)
        history.append(loss)

        # 발산하여 숫자가 무한대가 되면 중단
        if not np.isfinite(loss):
            break

    return a, b, np.array(history)


# TODO: 새로운 learning rate를 추가해 비교해 보세요.
for lr_test in [0.0003, 0.003, 0.03]:
    a_test, b_test, hist = train_linear_regression(lr_test)

    print(
        f"lr={lr_test:<7} "
        f"a={a_test:>10.4f}, "
        f"b={b_test:>10.4f}, "
        f"final MSE={hist[-1]:.4f}"
    )


## 6. 입력 변수가 여러 개인 경우

지금까지는 공부 시간 하나만 사용했습니다. 이제 입력 변수를 하나 더 추가해 봅니다.

- $x_1$: 공부 시간
- $x_2$: 과거 시험 횟수
- $y$: 시험 점수

다중 선형 회귀 모델은 다음과 같습니다.

$$
\hat{y}
=
a_1x_1 + a_2x_2 + b
$$

### 확인할 내용
입력 변수가 두 개가 되면 회귀선이 아니라 **회귀 평면(regression plane)**을 학습하게 됩니다.


In [ ]:
x1 = np.array([1, 3, 5, 7, 9], dtype=float)
x2 = np.array([0, 1, 5, 2, 4], dtype=float)
y_multi = np.array([75, 77, 85, 83, 90], dtype=float)

fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(x1, x2, y_multi, s=50)
ax.set_xlabel("Study Time")
ax.set_ylabel("Past Exams")
ax.set_zlabel("Exam Score")
ax.set_title("Multiple Linear Regression Data")
plt.show()

In [ ]:
# 초기 파라미터
a1 = 0.0
a2 = 0.0
b = 0.0

lr = 0.0003
epochs = 20001
n = len(x1)

for epoch in range(epochs):
    # 예측
    y_pred = a1 * x1 + a2 * x2 + b

    # 오차
    error = y_multi - y_pred

    # 각 파라미터의 gradient
    a1_diff = -(2 / n) * np.sum(x1 * error)
    a2_diff = -(2 / n) * np.sum(x2 * error)
    b_diff = -(2 / n) * np.sum(error)

    # 파라미터 업데이트
    a1 -= lr * a1_diff
    a2 -= lr * a2_diff
    b -= lr * b_diff

    if epoch % 4000 == 0:
        print(f"epoch: {epoch:5d}, a1: {a1:.6f}, a2: {a2:.6f}, b: {b:.6f}")

y_pred_multi = a1 * x1 + a2 * x2 + b

print("실제값:", y_multi)
print("예측값:", y_pred_multi)


In [ ]:
x1_grid = np.linspace(x1.min(), x1.max(), 20)
x2_grid = np.linspace(x2.min(), x2.max(), 20)
X1g, X2g = np.meshgrid(x1_grid, x2_grid)
Yg = a1 * X1g + a2 * X2g + b

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(x1, x2, y_multi, s=55)
ax.plot_surface(X1g, X2g, Yg, alpha=0.35)
ax.set_xlabel("Study Time")
ax.set_ylabel("Past Exams")
ax.set_zlabel("Exam Score")
ax.set_title("Best-Fit Regression Plane")
plt.show()

> ### ✅ 체크포인트
> 입력 변수가 1개일 때는 직선, 입력 변수가 2개일 때는 평면이 됩니다.  
> 입력 변수가 더 많아져도 기본 아이디어는 동일합니다.

## 7. TensorFlow/Keras로 같은 문제 풀기

앞에서는 선형 회귀의 학습 과정을 직접 구현했습니다.

이제 같은 문제를 TensorFlow/Keras로 구현합니다.

Keras에서는 일반적으로 다음 세 단계로 모델을 학습합니다.

1. **모델 정의**: `Dense(1)`
2. **학습 방법 설정**: `optimizer`, `loss`
3. **학습 실행**: `model.fit(...)`

이 구조는 이후 신경망 실습에서도 반복해서 사용됩니다.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

tf.random.set_seed(0)

### 7.1 단순 선형 회귀

입력 변수가 하나이므로 각 샘플은 하나의 feature를 가집니다. Keras 입력은 `(batch, feature)` 형태가 되도록 `x_tf`를 `(N, 1)`로 구성합니다.

```python
Input(shape=(1,))
Dense(1, activation="linear")
```

`Dense(1)`의 가중치와 편향은 앞에서 사용한 $a$, $b$와 같은 역할을 합니다.

### 확인할 내용
Keras가 학습한 `weight`와 `bias`가 직접 계산한 값과 얼마나 가까운지 비교하세요.


In [ ]:
x_tf = np.array([1, 3, 5, 7, 9], dtype=np.float32).reshape(-1, 1)
y_tf = np.array([75, 77, 85, 83, 90], dtype=np.float32)

# 선형 회귀 모델
model = Sequential([
    Input(shape=(1,)),
    Dense(1, activation="linear")
])

# optimizer와 loss 설정
model.compile(
    optimizer="sgd",
    loss="mse"
)

# 학습
model.fit(
    x_tf,
    y_tf,
    epochs=2000,
    verbose=0
)

# 훈련 데이터에 대한 예측
y_pred_tf = model(x_tf, training=False).numpy().squeeze()

print("훈련 데이터 예측값:", y_pred_tf)
print("실제 훈련 레이블:    ", y_tf)

# Dense 층의 학습된 파라미터 확인
weights, bias = model.layers[0].get_weights()

print(f"학습된 가중치: {weights.squeeze():.4f}")
print(f"학습된 편향:   {bias.squeeze():.4f}")


In [ ]:
# 학습된 회귀선 확인
plt.scatter(x_tf.squeeze(), y_tf)
plt.plot(x_tf.squeeze(), y_pred_tf)
plt.xlabel("Study Time (hours)")
plt.ylabel("Exam Score")
plt.title("TensorFlow Linear Regression")
plt.grid(alpha=0.3)
plt.show()

# TODO: hours 값을 바꾸어 예측 결과를 확인해 보세요.
hours = 7

sample = np.array([[hours]], dtype=np.float32)

# 소수의 샘플을 예측할 때는 model(...)로 직접 호출
exp_score = model(sample, training=False).numpy().squeeze()

print(f"{hours}시간 공부했을 때의 예측 점수:", exp_score)


### 7.2 다중 선형 회귀

입력 변수가 두 개이므로 입력의 shape는 `(2,)`가 됩니다.

```python
Input(shape=(2,))
Dense(1, activation="linear")
```

### 직접 해보기 3
아래 코드에서

- `hours`
- `past_exams`

값을 바꾸어 예측 점수가 어떻게 달라지는지 확인하세요.


In [ ]:
x_multi_tf = np.array(
    [[1, 0],
     [3, 1],
     [5, 5],
     [7, 2],
     [9, 4]],
    dtype=np.float32
)

y_multi_tf = np.array(
    [75, 77, 85, 83, 90],
    dtype=np.float32
)

# 입력 변수가 2개인 선형 회귀 모델
model_multi = Sequential([
    Input(shape=(2,)),
    Dense(1, activation="linear")
])

model_multi.compile(
    optimizer="sgd",
    loss="mse"
)

model_multi.fit(
    x_multi_tf,
    y_multi_tf,
    epochs=2000,
    verbose=0
)

y_pred = model_multi(
    x_multi_tf,
    training=False
).numpy().squeeze()

print("훈련 데이터 예측값:", y_pred)
print("실제 훈련 레이블:    ", y_multi_tf)

# TODO: 두 입력값을 변경해 보세요.
hours = 6
past_exams = 4

sample = np.array(
    [[hours, past_exams]],
    dtype=np.float32
)

exp_score = model_multi(
    sample,
    training=False
).numpy().squeeze()

print("예측 점수:", exp_score)


## 8. 사용자 정의 손실 함수 만들기

Keras에서는 기본 제공 손실 함수뿐 아니라 직접 만든 함수도 사용할 수 있습니다.

아래 함수는 MSE와 같은 형태입니다.

$$
\mathcal{L}
=
\operatorname{mean}
\left(
(y-\hat{y})^2
\right)
$$

### 확인할 내용
`model.compile(loss=squared_2norm_loss)`처럼 사용자 정의 함수를 손실 함수로 전달할 수 있음을 확인하세요.

> 이후 다양한 학습 목표를 설계할 때 사용자 정의 손실 함수가 매우 중요하게 사용됩니다.


In [ ]:
# 사용자 정의 MSE 손실 함수
def squared_2norm_loss(y_true, y_pred):
    return tf.reduce_mean(
        tf.square(y_true - y_pred)
    )


model_custom = Sequential([
    Input(shape=(2,)),
    Dense(1, activation="linear")
])

model_custom.compile(
    optimizer="sgd",
    loss=squared_2norm_loss
)

model_custom.fit(
    x_multi_tf,
    y_multi_tf,
    epochs=2000,
    verbose=0
)

y_pred_custom = model_custom(
    x_multi_tf,
    training=False
).numpy().squeeze()

print("훈련 데이터 예측값:", y_pred_custom)
print("실제 훈련 레이블:    ", y_multi_tf)


## 9. 최종 실습

아래 문제를 순서대로 수행해 보세요.

### 기본
1. 경사하강법 예제에서 학습률을 변경하고 수렴 속도를 비교하세요.
2. TensorFlow 단순 선형 회귀 예제에서 `hours` 값을 변경하여 시험 점수를 예측하세요.
3. 다중 선형 회귀 예제에서 `hours`와 `past_exams`를 모두 변경해 보세요.

### 비교
4. 다음 세 방법으로 얻은 파라미터를 비교하세요.
   - 최소제곱법
   - 직접 구현한 경사하강법
   - Keras

### 도전
5. 사용자 정의 손실 함수를 **MAE**로 변경하고 결과를 비교하세요.

힌트:

$$
\mathrm{MAE}
=
\operatorname{mean}
\left(
|y-\hat{y}|
\right)
$$


## 10. 정리

이번 실습에서는 하나의 선형 회귀 문제를 여러 관점에서 살펴보았습니다.

| 방법 | 핵심 아이디어 |
|---|---|
| 최소제곱법 | 최적 파라미터를 수식으로 직접 계산 |
| 벡터화 | 반복문 대신 행렬 연산으로 계산 |
| 경사하강법 | 손실의 gradient를 이용해 반복적으로 파라미터 갱신 |
| 다중 선형 회귀 | 여러 입력 변수를 사용하도록 모델 확장 |
| TensorFlow/Keras | 모델, 손실 함수, optimizer를 이용해 학습 자동화 |
| 사용자 정의 손실 | 원하는 학습 목표를 직접 정의 |

### 꼭 기억할 것

선형 회귀의 학습 과정은 이후 신경망 학습과 같은 기본 구조를 가집니다.

**입력 → 모델 → 예측 → 손실 계산 → 파라미터 업데이트**

다음 강의부터는 이 구조를 더 복잡한 모델에 확장합니다.
